# DeepSparser - Synthetic Wavelet Demo

This notebook demonstrates training and inference on synthetic wavelet data.

**Prerequisites:** Run `python download_data.py` first to obtain the dataset.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path

from utils import load_config
from model.network import DeepSparser
from dataset.dataset_synthetic import TrainDataset, TestDataset

config = load_config('config/config_synthetic.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Training

Skip this cell if a trained checkpoint already exists.

In [ ]:
from train import train
train(config)

## Inference & Visualization

In [ ]:
# Load model
model = DeepSparser(config).to(device)
model.load_state_dict(torch.load(config.checkpoint_path, map_location=device))
model.eval()

# Load test data
test_data = np.load(Path(config.data_path) / 'testing_data.npz')
test_dataset = TestDataset(test_data['x'], test_data['n'], config)

# Visualize
indices = [10, 20, 30]
fig, axes = plt.subplots(len(indices), 3, figsize=(16, 3 * len(indices)), tight_layout=True)

for i, idx in enumerate(indices):
    s, n = test_dataset[idx]
    y = test_dataset.add_noise(s, n, snr_db=0)
    s_hat = model.denoise(y)
    
    axes[i, 0].plot(s, lw=0.8)
    axes[i, 1].plot(y, lw=0.8)
    axes[i, 2].plot(s_hat, lw=0.8)

axes[0, 0].set_title('Clean Signal')
axes[0, 1].set_title('Noisy Signal')
axes[0, 2].set_title('Denoised Signal')
plt.show()